In [7]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model="gpt-4.1-nano")
output_parser = StrOutputParser()


In [8]:
from dotenv import load_dotenv
import os
load_dotenv()
# os.getenv('OPENAI_API_KEY')

True

In [9]:
llm.invoke("Where is the capital of Korea? Answer me in Korean")

AIMessage(content='한국의 수도는 서울입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 7, 'prompt_tokens': 18, 'total_tokens': 25, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4.1-nano-2025-04-14', 'system_fingerprint': 'fp_38343a2f8f', 'id': 'chatcmpl-BmX1gJNJebSSpf98Pa0gYOoG5KF52', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--db1d7eba-7591-403a-9522-70d5b15a1894-0', usage_metadata={'input_tokens': 18, 'output_tokens': 7, 'total_tokens': 25, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

# 1. 나라 -> 음식 추천

In [24]:
food_prompt = PromptTemplate(
    template="""
    Please recommend the most famous food from {country}.    
    When answering, write full name of country and,
    please use the national flag and food of the answer as an emoji.
    And give a short explanation of the name of the food and one line
    """,
    input_variables=["country"]
)

food_chain = food_prompt | llm | output_parser

output_parser.invoke(llm.invoke(food_prompt.invoke({"country":"Japan"})))

'🇯🇵 Japan: Sushi 🍣  \nSushi means "sour rice" and refers to seasoned vinegared rice combined with various toppings or fillings, often including raw fish. It is a quintessential Japanese dish known worldwide.'

# 2. 음식 -> 레시피

In [27]:
recipe_prompt = PromptTemplate(
    template="""
    Write a detailed recipe for the food: {food}.
    Include ingredients and cooking instructions.
    When answering, please use the national flag and food of the answer as an emoji.
    """,
    input_variables=["food"]
)

recipe_chain = recipe_prompt | llm | output_parser
output_parser.invoke(llm.invoke(recipe_prompt.invoke({"food":"kimchi"})))

"Certainly! Here's a detailed recipe for making traditional Korean kimchi 🇰🇷🫓:\n\n### Ingredients:\n- 1 large Napa cabbage (about 2-3 pounds)\n- 1/4 cup sea salt or kosher salt\n- Water (for soaking)\n- 1 tablespoon grated ginger\n- 4-5 cloves garlic (minced)\n- 2 tablespoons fish sauce or soy sauce (for a vegetarian version)\n- 1 teaspoon sugar\n- 1 bunch green onions (cut into 2-inch pieces)\n- 1 medium carrot (julienned)\n- 1 daikon radish (julienned)\n- 3-4 tablespoons Korean red pepper flakes (gochugaru)\n- Optional: 1 tablespoon rice flour for a thicker paste\n\n### Instructions:\n\n1. **Prepare the Cabbage:**\n   - Cut the Napa cabbage lengthwise into quarters, then cut each quarter into halves or thirds, depending on size.\n   - Dissolve 1/4 cup salt in about 4 cups water.\n   - Submerge the cabbage pieces in the salty water, making sure they are fully submerged. Use a plate or a weight to keep them underwater.\n   - Leave to soak for 2 hours, flipping occasionally to ensure ev

# 3. 체인 연결

In [26]:
from langchain_core.runnables import RunnablePassthrough

final_chain = {"country": RunnablePassthrough()} | {"food": food_chain} | recipe_chain


# 4. 출력

In [32]:
# 나라 이름을 넣으면 대표 음식과 레시피를 자동 생성
result = final_chain.invoke("China")
print(result)


🇨🇳🦆🍽️ Peking Duck Recipe

**Introduction:**  
Peking Duck is a celebrated Chinese dish renowned for its crispy skin and tender meat. Traditionally served with thin pancakes, hoisin sauce, and sliced scallions, it’s a show-stopping dish that captures the rich culinary heritage of Beijing.

**Ingredients:**

*For the Duck:*  
- 1 whole duck (about 5-6 pounds), cleaned and patted dry  
- 2 tablespoons maltose syrup or honey  
- 1 tablespoon soy sauce  
- 1 tablespoon Shaoxing wine (Chinese cooking wine)  
- 2 teaspoons five-spice powder  
- 1 teaspoon salt  
- 4-5 slices ginger  
- 2-3 green onions (scallions)  

*For the Pancakes and Serving:*  
- Thin Chinese pancakes or spring roll wrappers  
- Hoisin sauce or sweet bean sauce  
- Sliced scallions  
- Julienne cucumber (optional)  

**Cooking Instructions:**

1. **Preparation of the Duck:**  
   - Remove any excess fat from the cavity and pat the duck dry thoroughly with paper towels.  
   - Mix soy sauce, Shaoxing wine, five-spice pow